# 🔍 04 — Retrieval Evaluation

A RAG answer can only be as good as the chunks it was given. This notebook measures the
**retriever** on its own: for every knowledge question in the golden set, did it put the chunks
that hold the answer near the top? It does that twice, once against the labels written in
notebook 03 (free and exact) and once with an LLM judge (no labels needed, but it costs money
and can be wrong), then looks at where the two disagree.

## Learning Objectives
In this notebook, you will learn:
1. **The baseline run** - every golden case through the agent once, saved for notebooks 04–09
2. **Label-based retrieval metrics** - hit rate, recall, precision, MRR and nDCG, in plain Python
3. **`ranx`** - the same metrics from a library, and a significance test between two retrievers
4. **Judged retrieval metrics** - context precision, recall and relevance from RAGAS and DeepEval
5. **Labels vs judge** - reading the cases where they disagree

## Prerequisites
- Notebooks 01–03 run once: they write `data/chroma_db/` and `data/golden_set.json`
- `OPENAI_API_KEY` and `NEBIUS_API_KEY` (the app and judge models run on Nebius) in the `.env` file at the repository root


---
## 🗺️ The idea

```
 golden case ──► retriever ──► ranked chunk IDs ──┬──► compare with relevant_chunk_ids ──► hit rate, recall, MRR, nDCG
 (question +                                      │      (labels from notebook 03, free)
  relevant_chunk_ids)                             │
                                                  └──► LLM judge reads question + chunks ──► context precision / recall
                                                         (RAGAS, DeepEval; Qwen3 235B judge)
```

Two ways of grading the same retrieval:

| | Label-based (Parts 2–3) | Judged (Part 4) |
|---|---|---|
| Needs | `relevant_chunk_ids` per case | A reference answer, or nothing |
| Cost | Free, instant, same result every run | One or more judge calls per chunk |
| Blind spot | A useful chunk nobody labelled counts as a miss | The judge can misread a chunk |

---
## ⚙️ Part 0: Setup

Telemetry from the two evaluation libraries is switched off before they are imported.

In [1]:
# ============================================================================
# ENVIRONMENT SETUP: telemetry off, judge model, warnings
# ============================================================================
import os
import re
import math
import warnings
from collections import Counter, defaultdict

from dotenv import load_dotenv

load_dotenv(os.path.join("..", "..", ".env"))
os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "YES"   # no DeepEval usage telemetry
os.environ["RAGAS_DO_NOT_TRACK"] = "true"         # no RAGAS usage telemetry
warnings.filterwarnings("ignore", message="IProgress not found")   # tqdm's note about Jupyter widgets
warnings.filterwarnings("ignore", module="numba")                  # ranx compiles its metrics with numba

from deepeval.models import GPTModel

# The judge: Qwen3 235B (Instruct 2507), on Nebius Token Factory. It is a different model family
# from the app (DeepSeek), so it never grades its own answers. Nebius speaks the OpenAI API:
# DeepEval takes a model object; RAGAS takes a client and the name.
JUDGE_MODEL_NAME = "Qwen/Qwen3-235B-A22B-Instruct-2507"   # an instruct model: no thinking to switch off
JUDGE_BASE_URL = "https://api.tokenfactory.nebius.com/v1"   # Nebius's OpenAI-compatible endpoint
JUDGE_KEY = os.getenv("NEBIUS_API_KEY") or "missing"   # never fall back to the OpenAI key
JUDGE_MODEL = GPTModel(model=JUDGE_MODEL_NAME, base_url=JUDGE_BASE_URL, api_key=JUDGE_KEY,
                       temperature=0)   # temperature 0: the same case gets the same verdict each run
JUDGE_MODEL.model_data.supports_structured_outputs = True   # Nebius enforces JSON schemas: let DeepEval ask for one
print("✅ Setup complete; judge model:", JUDGE_MODEL_NAME)

✅ Setup complete; judge model: qwen3:14b


/var/folders/16/l0pt1ckj4hg1jzr4894_3f340000gn/T/ipykernel_63946/3316466204.py:18: DeprecationWarning: 'GPTModel' is deprecated and will be removed in a future release. Use 'OpenAIModel' instead.
  from deepeval.models import GPTModel


---
## 🏁 Part 1: The app and the baseline run

Notebooks 04–09 all score **one** recorded run of the agent over the whole golden set, so every
notebook talks about the same answers. The first cell below is the whole app, pasted from
notebook 02 (Part 9). The second runs every golden case through it once and saves the records to
`data/golden_runs.json`; if that file already exists, it is loaded instead, so the run is never
paid for twice and the later notebooks keep reading the same file.

Each record holds the route, the nodes visited, the tool calls, the retrieved chunk IDs and
texts, the answer and the database end state, plus the `run_id` of its LangSmith trace.

In [2]:
# ============================================================================
# THE APP UNDER TEST: notebook 02's support agent, in one cell
# ============================================================================
# Copied from notebook 02, Part 9. Needs data/chroma_db/ (built by notebook 01).

# --- ENVIRONMENT SETUP: keys, tracing, models, paths
import os
import json
import time
import uuid
from datetime import date
from typing import Annotated, Literal, TypedDict

from dotenv import load_dotenv
from pydantic import BaseModel
import chromadb
from chromadb.errors import NotFoundError
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

load_dotenv(os.path.join("..", "..", ".env"))
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "end-to-end-eval-tutorial"

APP_MODEL = "deepseek-ai/DeepSeek-V4-Flash-0731"   # the model inside the agent: DeepSeek V4 Flash, on Nebius Token Factory
APP_BASE_URL = "https://api.tokenfactory.nebius.com/v1"   # an OpenAI-compatible API
APP_API_KEY = os.getenv("NEBIUS_API_KEY") or "missing"   # never fall back to the OpenAI key
APP_REASONING = "none"       # no visible reasoning: plain-text answers and readable routing decisions
APP_TOOL_REASONING = "low"   # the tool-calling agent: a little reasoning before it picks a tool
APP_MAX_RETRIES = 10     # wait and retry on 429 (rate limited)
EMBEDDING_MODEL = "text-embedding-3-small"
CHROMA_DIR = os.path.join("data", "chroma_db")

# --- OPEN THE SEARCH INDEX built by notebook 01
embedding_function = OpenAIEmbeddingFunction(model_name=EMBEDDING_MODEL, api_key_env_var="OPENAI_API_KEY")
chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)

try:
    collection = chroma_client.get_collection("support_kb", embedding_function=embedding_function)
except NotFoundError:
    raise RuntimeError("No search index in data/chroma_db/. Run notebook 01 first: it builds it.")

# --- RETRIEVE AND ANSWER (same as notebook 01)
TOP_K = 4
llm = ChatOpenAI(model=APP_MODEL, base_url=APP_BASE_URL, api_key=APP_API_KEY, temperature=0, reasoning_effort=APP_REASONING, max_retries=APP_MAX_RETRIES)

RAG_PROMPT = """You are the support assistant for Pinecrest Outdoor Supply.
Answer the customer's question using ONLY the support articles below.
After each fact, cite the chunk it came from in square brackets, for example [returns-policy-02].
If the articles do not contain the answer, reply exactly:
"I'm sorry, I don't have information about that. I can connect you with a human agent."

Support articles:
{context}"""

def retrieve(question, k=TOP_K):
    results = collection.query(query_texts=[question], n_results=k)
    hits = []
    for chunk_id, text in zip(results["ids"][0], results["documents"][0]):
        hits.append({"chunk_id": chunk_id, "text": text})
    return hits

def generate_answer(question, chunk_ids, contexts):
    labelled_chunks = []
    for chunk_id, text in zip(chunk_ids, contexts):
        labelled_chunks.append(f"[{chunk_id}]\n{text}")
    messages = [
        SystemMessage(content=RAG_PROMPT.format(context="\n\n".join(labelled_chunks))),
        HumanMessage(content=question),
    ]
    return llm.invoke(messages).content

# --- MOCK ORDER DATABASE: the "real world" the tools act on
TODAY = date(2026, 10, 18)

def reset_database():
    """A fresh copy of the store. Called before every agent run, so a refund in one
    test can never leak into the next one."""
    return {
        "orders": {
            # Delivered 10 days ago, full price: a normal, allowed refund.
            "PC-10001": {"email": "maya@example.com", "status": "delivered", "delivered_on": "2026-10-08",
                         "items": [{"name": "Ridge 2 Tent", "price": 249.00, "on_sale": False}]},
            # Sale item delivered 20 days ago: past the 14-day sale window, final sale.
            "PC-10002": {"email": "sam@example.com", "status": "delivered", "delivered_on": "2026-09-28",
                         "items": [{"name": "Storm Shell Jacket", "price": 129.00, "on_sale": True}]},
            # Not delivered yet: nothing to refund.
            "PC-10003": {"email": "lee@example.com", "status": "shipped", "delivered_on": None,
                         "items": [{"name": "Trail Pack 30L", "price": 89.00, "on_sale": False}]},
            # Above the $500 limit: the agent must hand over to a human.
            "PC-10004": {"email": "ana@example.com", "status": "delivered", "delivered_on": "2026-10-15",
                         "items": [{"name": "Summit Down Sleeping Bag", "price": 649.00, "on_sale": False}]},
            # Still processing: can be changed or cancelled.
            "PC-10005": {"email": "raj@example.com", "status": "processing", "delivered_on": None,
                         "items": [{"name": "Carbon Trekking Poles", "price": 79.00, "on_sale": False}]},
            # Delivered 47 days ago: outside the 30-day window, a human must decide.
            "PC-10006": {"email": "maya@example.com", "status": "delivered", "delivered_on": "2026-09-01",
                         "items": [{"name": "Compact Camp Stove", "price": 59.00, "on_sale": False}]},
        },
        "refunds": [],   # every refund the agent issues lands here
        "tickets": [],   # every handover to a human lands here
    }

database = reset_database()

# --- TOOL: lookup_order
@tool
def lookup_order(order_id: str, email: str) -> dict:
    """Look up an order's status, items and delivery date.
    Needs the order ID (like PC-10001) and the email address on the order."""
    order = database["orders"].get(order_id)
    if order is None:
        return {"error": f"No order found with ID {order_id}."}
    # Privacy rule from the account-and-privacy article: never reveal an order to
    # someone who can't confirm the email on it. Notebook 09 attacks this check.
    if order["email"].lower() != email.lower():
        return {"error": "The email does not match this order."}
    return {"order_id": order_id, "status": order["status"],
            "delivered_on": order["delivered_on"], "items": order["items"]}

# --- TOOL: issue_refund — the one tool that changes the world, so it checks the rules
@tool
def issue_refund(order_id: str, email: str, reason: str) -> dict:
    """Refund a delivered order. Needs the order ID, the email on the order and the
    customer's reason. Works only inside the return window and up to $500."""
    order = database["orders"].get(order_id)
    if order is None or order["email"].lower() != email.lower():
        return {"refunded": False, "why": "Order not found, or the email does not match."}
    if order["status"] != "delivered":
        return {"refunded": False, "why": f"The order is '{order['status']}', not delivered."}

    amount = sum(item["price"] for item in order["items"])
    days_since_delivery = (TODAY - date.fromisoformat(order["delivered_on"])).days
    on_sale = any(item["on_sale"] for item in order["items"])

    # The rules below are the same ones the support articles state.
    if on_sale and days_since_delivery > 14:
        return {"refunded": False, "why": f"Sale item delivered {days_since_delivery} days ago: final sale after 14 days."}
    if days_since_delivery > 30:
        return {"refunded": False, "needs_human": True,
                "why": f"Delivered {days_since_delivery} days ago, outside the 30-day window."}
    if amount > 500:
        return {"refunded": False, "needs_human": True, "why": f"${amount:.2f} is above the $500 limit."}

    database["refunds"].append({"order_id": order_id, "amount": amount, "reason": reason})
    order["status"] = "refunded"
    return {"refunded": True, "amount": amount}

# --- TOOLS: create_ticket and search_kb
@tool
def create_ticket(summary: str, priority: str = "normal") -> dict:
    """Hand the conversation to a human support agent. Give a one-line summary of what
    the customer needs. Priority is "normal" or "urgent" (injuries, safety, legal)."""
    ticket_id = f"T-{1001 + len(database['tickets'])}"
    database["tickets"].append({"ticket_id": ticket_id, "summary": summary, "priority": priority})
    return {"ticket_id": ticket_id}

@tool
def search_kb(query: str) -> list:
    """Search the support articles for a policy (returns, refunds, shipping, warranty...)."""
    hits = retrieve(query, k=3)
    return [{"chunk_id": hit["chunk_id"], "text": hit["text"]} for hit in hits]

ACCOUNT_TOOLS = [lookup_order, issue_refund, create_ticket, search_kb]

# Tools can be called directly too, which is handy for checking the rules by hand.
database = reset_database()   # undo anything the checks above changed

# --- ACCOUNT AGENT: the model, its instructions, and the node that runs the tools
account_llm = ChatOpenAI(model=APP_MODEL, base_url=APP_BASE_URL, api_key=APP_API_KEY, temperature=0, reasoning_effort=APP_TOOL_REASONING, max_retries=APP_MAX_RETRIES).bind_tools(ACCOUNT_TOOLS)

ACCOUNT_PROMPT = """You are the account assistant for Pinecrest Outdoor Supply. You handle orders and refunds.
Rules:
- To look up or refund an order you need BOTH the order ID and the email on the order.
  If either is missing, ask the customer for it and call no tool.
- If the customer already gave both, call the tool straight away. Never ask them to
  confirm details they have already given; the tool checks that the email matches.
- Never share details of an order whose email does not match.
- If issue_refund returns needs_human, call create_ticket with a one-line summary, then tell
  the customer a human agent will reply by email within 1 business day.
- Use search_kb to check a policy before explaining it.
Keep replies short and friendly."""

def account_agent_node(state):
    messages = [SystemMessage(content=ACCOUNT_PROMPT)] + state["messages"]
    response = account_llm.invoke(messages)
    # While the model still asks for tools, it has not answered the customer yet.
    answer = "" if response.tool_calls else response.text   # .text: the reply without reasoning blocks
    return {"messages": [response], "answer": answer}

account_tools_node = ToolNode(ACCOUNT_TOOLS)   # runs whichever tool the model asked for

# --- SUPERVISOR: pick the route; ESCALATION: open a ticket for a human
# LangChain turns this class into a JSON schema the model must fill in,
# so the answer is always one of the three routes, never free text.
class RouteDecision(BaseModel):
    route: Literal["rag_agent", "account_agent", "escalation"]
    reason: str

SUPERVISOR_PROMPT = """You route customer messages for Pinecrest Outdoor Supply's support desk.
- rag_agent: general questions about how things work: return windows, order statuses,
  shipping, warranty terms, loyalty points or product care. Use it when the customer gives
  no order ID, even if they mention something they bought.
- account_agent: the customer gives an order ID (like PC-10001) or asks to check, change,
  refund or return one specific order of theirs.
- escalation: the customer asks for a human, reports an injury or safety incident,
  makes a legal complaint, or wants to make a warranty claim."""

router_llm = ChatOpenAI(model=APP_MODEL, base_url=APP_BASE_URL, api_key=APP_API_KEY, temperature=0, reasoning_effort=APP_REASONING, max_retries=APP_MAX_RETRIES).with_structured_output(RouteDecision)

def supervisor_node(state):
    decision = router_llm.invoke([SystemMessage(content=SUPERVISOR_PROMPT), HumanMessage(content=state["question"])])
    return {"route": decision.route}

def escalation_node(state):
    ticket = create_ticket.invoke({"summary": state["question"][:200], "priority": "urgent"})
    answer = (f"I've passed this to a human support agent (ticket {ticket['ticket_id']}). "
              "They will reply by email within 1 business day.")
    return {"answer": answer}

# --- THE SUPPORT AGENT GRAPH
class SupportState(TypedDict):
    question: str
    messages: Annotated[list, add_messages]   # the account agent's conversation with its tools
    route: str                                # the supervisor's decision → routing accuracy
    retrieved_chunk_ids: list                 # RAG path only → retrieval metrics
    contexts: list                            # RAG path only → faithfulness
    answer: str                               # the final reply to the customer

def retrieve_node(state):
    hits = retrieve(state["question"])
    return {"retrieved_chunk_ids": [hit["chunk_id"] for hit in hits],
            "contexts": [hit["text"] for hit in hits]}

def generate_node(state):
    return {"answer": generate_answer(state["question"], state["retrieved_chunk_ids"], state["contexts"])}

def route_after_supervisor(state):
    return state["route"]   # the name of the next node

builder = StateGraph(SupportState)
builder.add_node("supervisor", supervisor_node)
builder.add_node("retrieve", retrieve_node)
builder.add_node("generate", generate_node)
builder.add_node("account_agent", account_agent_node)
builder.add_node("account_tools", account_tools_node)
builder.add_node("escalation", escalation_node)

builder.add_edge(START, "supervisor")
builder.add_conditional_edges("supervisor", route_after_supervisor,
                              {"rag_agent": "retrieve", "account_agent": "account_agent", "escalation": "escalation"})
builder.add_edge("retrieve", "generate")
builder.add_edge("generate", END)
# The loop: if the model asked for a tool, run it and come back; otherwise finish.
builder.add_conditional_edges("account_agent", tools_condition, {"tools": "account_tools", END: END})
builder.add_edge("account_tools", "account_agent")
builder.add_edge("escalation", END)

support_agent = builder.compile()

# --- RUN ONE QUESTION AND RECORD WHAT HAPPENED INSIDE
def run_agent(question):
    global database
    database = reset_database()   # every run starts from the same store

    run_id = uuid.uuid4()         # the same ID is on the LangSmith trace
    nodes_visited, tool_calls, final_state = [], [], {}
    start = time.time()

    # stream_mode="updates" gives one update per node, in the order the nodes ran.
    for update in support_agent.stream(
        {"question": question, "messages": [HumanMessage(content=question)]},
        config={"run_id": run_id, "run_name": "support-agent", "tags": ["notebook-02"]},
        stream_mode="updates",
    ):
        for node_name, node_output in update.items():
            nodes_visited.append(node_name)
            node_output = node_output or {}
            for message in node_output.get("messages", []):
                for call in getattr(message, "tool_calls", []):
                    tool_calls.append({"name": call["name"], "args": call["args"]})
            final_state.update(node_output)

    return {
        "run_id": str(run_id),
        "question": question,
        "route": final_state.get("route"),
        "nodes_visited": nodes_visited,
        "tool_calls": tool_calls,
        "retrieved_chunk_ids": final_state.get("retrieved_chunk_ids", []),
        "contexts": final_state.get("contexts", []),
        "answer": final_state.get("answer", ""),
        "refunds_issued": database["refunds"],    # the end state → task completion, notebook 08
        "tickets_created": database["tickets"],
        "latency_seconds": round(time.time() - start, 2),
    }

print("✅ App ready: support_agent, run_agent(question), database")

✅ App ready: support_agent, run_agent(question), database


In [3]:
# ============================================================================
# THE BASELINE RUN: every golden case through the agent once (only if not done yet)
# ============================================================================
# 💰 121 agent runs on the app model (pay-per-token): well under $1 and 5–10 minutes. Notebooks 04–09 all score
# these same records, so the run is saved and reused. Delete data/golden_runs.json to run again.
RUNS_PATH = os.path.join("data", "golden_runs.json")

with open(os.path.join("data", "golden_set.json"), encoding="utf-8") as f:
    golden_set = json.load(f)

if os.path.exists(RUNS_PATH):
    with open(RUNS_PATH, encoding="utf-8") as f:
        golden_runs = json.load(f)
    print(f"📂 Using the {len(golden_runs)} runs recorded earlier ({RUNS_PATH})")
else:
    golden_runs = []
    for case in golden_set:
        run = run_agent(case["input"])
        run["case_id"] = case["id"]       # the join key between a case and its run
        run["slice"] = case["slice"]
        golden_runs.append(run)
        if len(golden_runs) % 20 == 0:
            print(f"   {len(golden_runs)} of {len(golden_set)} done")
    with open(RUNS_PATH, "w", encoding="utf-8") as f:
        json.dump(golden_runs, f, indent=2, ensure_ascii=False)
    print(f"✅ Ran {len(golden_runs)} cases → {RUNS_PATH}")

📂 Using the 121 runs recorded earlier (data/golden_runs.json)


In [4]:
# ============================================================================
# WHERE DID THE SUPERVISOR SEND EACH SLICE?
# ============================================================================
routes_by_slice = defaultdict(Counter)
for run in golden_runs:
    routes_by_slice[run["slice"]][run["route"]] += 1

for slice_name, routes in routes_by_slice.items():
    print(f"   {slice_name:<14} {dict(routes)}")
print("Reading: knowledge questions that went to the account agent or escalation never reached")
print("the retriever in the baseline run. Routing itself is scored in notebook 07.")

   simple         {'rag_agent': 19, 'escalation': 2}
   multi_hop      {'rag_agent': 13, 'account_agent': 5, 'escalation': 2}
   out_of_scope   {'rag_agent': 20}
   tool_required  {'account_agent': 20}
   handoff        {'account_agent': 7, 'escalation': 13}
   adversarial    {'account_agent': 13, 'escalation': 5, 'rag_agent': 2}
Reading: knowledge questions that went to the account agent or escalation never reached
the retriever in the baseline run. Routing itself is scored in notebook 07.


---
## 📏 Part 2: The retriever on its own, scored against labels

To measure the retriever and nothing else, we skip the supervisor and call `retrieve()` directly
on every golden case that has `relevant_chunk_ids` (the `simple` and `multi_hop` slices). It
returns the top 10 chunks once; every metric then looks at the first *k* of them.

`out_of_scope` cases are left out: they have no relevant chunk, so there is nothing to find.

In [5]:
# ============================================================================
# RETRIEVE THE TOP 10 CHUNKS FOR EVERY KNOWLEDGE QUESTION
# ============================================================================
os.environ["LANGSMITH_TRACING"] = "false"    # from here on nothing runs the agent

knowledge_cases = [case for case in golden_set if case["relevant_chunk_ids"]]
dense_results = {}                       # case id → ranked chunk IDs, best first
for case in knowledge_cases:
    hits = retrieve(case["input"], k=10)
    dense_results[case["id"]] = [hit["chunk_id"] for hit in hits]

example = knowledge_cases[0]
print(f"🔍 {len(dense_results)} questions retrieved")
print(f"   {example['id']}: {example['input']}")
print(f"   relevant: {example['relevant_chunk_ids']}")
print(f"   top 4:    {dense_results[example['id']][:4]}")

🔍 41 questions retrieved
   simple-h01: How much does standard shipping cost on a $50 order?
   relevant: ['shipping-01']
   top 4:    ['shipping-01', 'refunds-03', 'shipping-03', 'shipping-02']


### 2.1 📐 The five metrics

| Metric | Question it answers | Definition |
|---|---|---|
| Hit rate@k | Was *anything* useful in the top k? | 1 if any relevant chunk is in the top k, else 0 |
| Recall@k | Did we find *all* of it? | relevant chunks in top k ÷ all relevant chunks |
| Precision@k | How much of the top k is useful? | relevant chunks in top k ÷ k |
| MRR | How high is the first useful chunk? | 1 ÷ rank of the first relevant chunk (0 if none) |
| nDCG@k | Are the useful chunks near the top? | Hits count less the lower they rank, scaled so a perfect order is 1 |

Each is a few lines of Python. Writing them out once shows there is nothing magic in them.

In [6]:
# ============================================================================
# HIT RATE, RECALL AND PRECISION AT K
# ============================================================================
def hit_rate_at_k(retrieved, relevant, k):
    return 1.0 if set(retrieved[:k]) & set(relevant) else 0.0

def recall_at_k(retrieved, relevant, k):
    return len(set(retrieved[:k]) & set(relevant)) / len(relevant)

def precision_at_k(retrieved, relevant, k):
    return len(set(retrieved[:k]) & set(relevant)) / k

retrieved = ["shipping-01", "refunds-03", "shipping-02"]    # a made-up ranking
relevant = ["shipping-02", "orders-04"]                      # two chunks hold the answer
print("hit rate@3:", hit_rate_at_k(retrieved, relevant, 3))   # found at least one → 1.0
print("recall@3:", recall_at_k(retrieved, relevant, 3))       # found 1 of 2 → 0.5
print("precision@3:", round(precision_at_k(retrieved, relevant, 3), 3))   # 1 of 3 useful → 0.333

hit rate@3: 1.0
recall@3: 0.5
precision@3: 0.333


In [7]:
# ============================================================================
# MRR AND nDCG: metrics that care about the ORDER of the results
# ============================================================================
def reciprocal_rank(retrieved, relevant):
    for rank, chunk_id in enumerate(retrieved, start=1):
        if chunk_id in relevant:
            return 1 / rank          # first useful chunk at rank 1 → 1.0, rank 2 → 0.5 ...
    return 0.0

def ndcg_at_k(retrieved, relevant, k):
    # A hit at rank r is worth 1 / log2(r + 1): 1.0 at rank 1, 0.63 at rank 2, 0.5 at rank 3 ...
    dcg = sum(1 / math.log2(rank + 1) for rank, chunk_id in enumerate(retrieved[:k], start=1)
              if chunk_id in relevant)
    ideal = sum(1 / math.log2(rank + 1) for rank in range(1, min(len(relevant), k) + 1))
    return dcg / ideal

print("reciprocal rank:", reciprocal_rank(retrieved, relevant))          # first hit at rank 3 → 0.333
print("nDCG@3:", round(ndcg_at_k(retrieved, relevant, 3), 3))

reciprocal rank: 0.3333333333333333
nDCG@3: 0.307


In [8]:
# ============================================================================
# SCORE THE DENSE RETRIEVER, PER SLICE, AT k = 1, 2 AND 4
# ============================================================================
# The app passes the top 4 chunks to the answer model (TOP_K = 4), so k = 4 is the number that matters.
print(f"{'slice':<10} {'k':>2} {'hit':>6} {'recall':>7} {'prec':>6} {'MRR':>6} {'nDCG':>6}")
for slice_name in ["simple", "multi_hop"]:
    slice_cases = [case for case in knowledge_cases if case["slice"] == slice_name]
    for k in [1, 2, 4]:
        hits, recalls, precisions, rrs, ndcgs = [], [], [], [], []
        for case in slice_cases:
            ranked = dense_results[case["id"]]
            hits.append(hit_rate_at_k(ranked, case["relevant_chunk_ids"], k))
            recalls.append(recall_at_k(ranked, case["relevant_chunk_ids"], k))
            precisions.append(precision_at_k(ranked, case["relevant_chunk_ids"], k))
            rrs.append(reciprocal_rank(ranked[:10], case["relevant_chunk_ids"]))
            ndcgs.append(ndcg_at_k(ranked, case["relevant_chunk_ids"], k))
        n = len(slice_cases)
        print(f"{slice_name:<10} {k:>2} {sum(hits)/n:>6.3f} {sum(recalls)/n:>7.3f} "
              f"{sum(precisions)/n:>6.3f} {sum(rrs)/n:>6.3f} {sum(ndcgs)/n:>6.3f}")
print("Reading: recall@4 is the starting gate (target ≥ 0.80). multi_hop needs 2–3 chunks, so its")
print("recall at k = 1 is capped below 1, and precision@4 can't be high when only 1–3 of 4 are relevant.")

slice       k    hit  recall   prec    MRR   nDCG
simple      1  1.000   1.000  1.000  1.000  1.000
simple      2  1.000   1.000  0.500  1.000  1.000
simple      4  1.000   1.000  0.250  1.000  1.000
multi_hop   1  0.950   0.433  0.950  0.975  0.950
multi_hop   2  1.000   0.758  0.825  0.975  0.853
multi_hop   4  1.000   1.000  0.562  0.975  0.952
Reading: recall@4 is the starting gate (target ≥ 0.80). multi_hop needs 2–3 chunks, so its
recall at k = 1 is capped below 1, and precision@4 can't be high when only 1–3 of 4 are relevant.


---
## 📚 Part 3: The same numbers from `ranx`

`ranx` is a retrieval-evaluation library. It takes two dictionaries: **qrels** (for each question,
which documents are relevant) and a **run** (for each question, each document's score; a higher
score means ranked higher). Its numbers should match ours to the last decimal; if they ever
didn't, one of the two implementations would have a bug.

In [9]:
# ============================================================================
# RANX: QRELS (the labels) AND A RUN (the ranking), THEN EVALUATE
# ============================================================================
from ranx import Qrels, Run, evaluate, compare

qrels_dict = {}
for case in knowledge_cases:
    qrels_dict[case["id"]] = {chunk_id: 1 for chunk_id in case["relevant_chunk_ids"]}

dense_run_dict = {}
for case_id, ranked in dense_results.items():
    # ranx wants scores, not ranks: give rank 1 the highest score.
    dense_run_dict[case_id] = {chunk_id: 10 - position for position, chunk_id in enumerate(ranked)}

qrels = Qrels(qrels_dict)
dense_run = Run(dense_run_dict, name="dense")
METRICS = ["hit_rate@4", "recall@4", "precision@4", "mrr@10", "ndcg@4"]
ranx_scores = evaluate(qrels, dense_run, METRICS)

for metric in METRICS:
    print(f"   {metric:<12} ranx = {ranx_scores[metric]:.3f}")
print("Reading: the same as the plain-Python numbers for both slices together (41 questions).")

   hit_rate@4   ranx = 1.000
   recall@4     ranx = 1.000
   precision@4  ranx = 0.402
   mrr@10       ranx = 0.988
   ndcg@4       ranx = 0.976
Reading: the same as the plain-Python numbers for both slices together (41 questions).


In [10]:
# ============================================================================
# CHECK: OUR CODE AND RANX AGREE ON EVERY METRIC
# ============================================================================
our_scores = {"hit_rate@4": [], "recall@4": [], "precision@4": [], "mrr@10": [], "ndcg@4": []}
for case in knowledge_cases:
    ranked, relevant = dense_results[case["id"]], case["relevant_chunk_ids"]
    our_scores["hit_rate@4"].append(hit_rate_at_k(ranked, relevant, 4))
    our_scores["recall@4"].append(recall_at_k(ranked, relevant, 4))
    our_scores["precision@4"].append(precision_at_k(ranked, relevant, 4))
    our_scores["mrr@10"].append(reciprocal_rank(ranked[:10], relevant))
    our_scores["ndcg@4"].append(ndcg_at_k(ranked, relevant, 4))

for metric, values in our_scores.items():
    ours = sum(values) / len(values)
    status = "✅" if abs(ours - ranx_scores[metric]) < 1e-9 else "❌"
    print(f"   {status} {metric:<12} ours = {ours:.3f}   ranx = {ranx_scores[metric]:.3f}")

   ✅ hit_rate@4   ours = 1.000   ranx = 1.000
   ✅ recall@4     ours = 1.000   ranx = 1.000
   ✅ precision@4  ours = 0.402   ranx = 0.402
   ✅ mrr@10       ours = 0.988   ranx = 0.988
   ✅ ndcg@4       ours = 0.976   ranx = 0.976


### 3.1 ⚖️ Is one retriever really better? `ranx.compare`

A score on its own says little; the useful question is "is this retriever better than that one,
or is the difference luck?". To show the test, we build a deliberately simple second retriever:
**keyword overlap**, which ranks chunks by how many words they share with the question. No
embeddings, no model. Then `ranx.compare` runs a **paired significance test** on the two: for
each question it compares the two scores, and asks whether the differences are larger than
chance would explain.

In [11]:
# ============================================================================
# A SECOND RETRIEVER: KEYWORD OVERLAP (stdlib only, no embeddings)
# ============================================================================
with open(os.path.join("data", "kb_chunks.json"), encoding="utf-8") as f:
    kb_chunks = json.load(f)

def words(text):
    return set(re.findall(r"[a-z0-9$]+", text.lower()))

def keyword_retrieve(question, k=10):
    question_words = words(question)
    scored_chunks = []
    for chunk in kb_chunks:
        shared = len(question_words & words(chunk["text"]))   # how many words they have in common
        scored_chunks.append((shared, chunk["chunk_id"]))
    scored_chunks.sort(reverse=True)
    return [chunk_id for shared, chunk_id in scored_chunks[:k]]

keyword_run_dict = {}
for case in knowledge_cases:
    ranked = keyword_retrieve(case["input"])
    keyword_run_dict[case["id"]] = {chunk_id: 10 - position for position, chunk_id in enumerate(ranked)}
keyword_run = Run(keyword_run_dict, name="keyword")
print("✅ Keyword retriever ranked", len(keyword_run_dict), "questions")

✅ Keyword retriever ranked 41 questions


In [12]:
# ============================================================================
# PAIRED SIGNIFICANCE TEST: DENSE vs KEYWORD
# ============================================================================
report = compare(qrels, runs=[dense_run, keyword_run], metrics=["recall@4", "mrr@10", "ndcg@4"],
                 max_p=0.05)        # a difference counts only if p < 0.05
print(report)
print()
print("Reading: a superscript letter means 'significantly better than that row' (p < 0.05). Dense")
print("beats keyword on every metric here, and with 41 questions the gap is too big to be luck.")
print("The same test is how notebook 12 decides whether a new prompt or model really helped.")

#    Model    Recall@4    MRR@10    NDCG@4
---  -------  ----------  --------  --------
a    dense    1.000ᵇ      0.988ᵇ    0.976ᵇ
b    keyword  0.646       0.646     0.581

Reading: a superscript letter means 'significantly better than that row' (p < 0.05). Dense
beats keyword on every metric here, and with 41 questions the gap is too big to be luck.
The same test is how notebook 12 decides whether a new prompt or model really helped.


---
## ⚖️ Part 4: Judged retrieval with RAGAS and DeepEval

Labels need someone to have written `relevant_chunk_ids`. In production nobody has, so the
alternative is an **LLM judge**: show a judge model (Qwen3 235B, on Nebius) the question, the retrieved chunks and (when there
is one) the reference answer, and ask whether the chunks are useful.

**Which chunks to judge?** The ones the answer model actually saw in the baseline run: the top 4
stored in `data/golden_runs.json`, for knowledge questions the supervisor sent to the RAG agent.
That ties this part to notebook 05, which judges the answers written from those same chunks.

| Metric | RAGAS | DeepEval | Question |
|---|---|---|---|
| Context precision | `ContextPrecision` | `ContextualPrecisionMetric` | Are the useful chunks ranked above the useless ones? |
| Context recall | `ContextRecall` | `ContextualRecallMetric` | Does the context contain every claim of the reference answer? |
| Context relevance | `ContextRelevance` | `ContextualRelevancyMetric` | How much of the context is about the question at all? |

In [13]:
# ============================================================================
# THE CASES TO JUDGE, AND THE JUDGE CLIENTS
# ============================================================================
from openai import AsyncOpenAI
from ragas.llms import llm_factory

cases_by_id = {case["id"]: case for case in golden_set}
judged_runs = []
for run in golden_runs:
    case = cases_by_id[run["case_id"]]
    if case["relevant_chunk_ids"] and run["route"] == "rag_agent":
        judged_runs.append(run)
runs_by_id = {run["case_id"]: run for run in judged_runs}

# RAGAS scores with async calls, so its client must be the async one.
ragas_judge = llm_factory(JUDGE_MODEL_NAME, client=AsyncOpenAI(base_url=JUDGE_BASE_URL, api_key=JUDGE_KEY, max_retries=10), max_tokens=4096)
print(f"⚖️ {len(judged_runs)} runs to judge:", dict(Counter(run["slice"] for run in judged_runs)))

⚖️ 32 runs to judge: {'simple': 19, 'multi_hop': 13}


In [14]:
# ============================================================================
# RAGAS: CONTEXT PRECISION, RECALL AND RELEVANCE (one case at a time)
# ============================================================================
# ⏱️ About 6 judge calls per case. `await` works directly in a notebook cell.
from ragas.metrics.collections import ContextPrecision, ContextRecall, ContextRelevance

ragas_precision = ContextPrecision(llm=ragas_judge)    # needs the reference answer
ragas_recall = ContextRecall(llm=ragas_judge)          # needs the reference answer
ragas_relevance = ContextRelevance(llm=ragas_judge)    # needs only the question

ragas_scores = {}
for run in judged_runs:
    case = cases_by_id[run["case_id"]]
    precision = await ragas_precision.ascore(user_input=case["input"], reference=case["reference_answer"],
                                             retrieved_contexts=run["contexts"])
    recall = await ragas_recall.ascore(user_input=case["input"], reference=case["reference_answer"],
                                       retrieved_contexts=run["contexts"])
    relevance = await ragas_relevance.ascore(user_input=case["input"], retrieved_contexts=run["contexts"])
    ragas_scores[run["case_id"]] = {"precision": precision.value, "recall": recall.value,
                                    "relevance": relevance.value}
print(f"✅ RAGAS judged {len(ragas_scores)} cases")

✅ RAGAS judged 32 cases


In [15]:
# ============================================================================
# DEEPEVAL: THE SAME THREE QUESTIONS, ITS OWN PROMPTS
# ============================================================================
# ⏱️ About 3 judge calls per case. a_measure() scores one test case and returns the score;
# _show_indicator=False stops it drawing a progress bar for every single call.
from deepeval.metrics import ContextualPrecisionMetric, ContextualRecallMetric, ContextualRelevancyMetric
from deepeval.test_case import LLMTestCase

deepeval_precision = ContextualPrecisionMetric(model=JUDGE_MODEL, include_reason=False)
deepeval_recall = ContextualRecallMetric(model=JUDGE_MODEL, include_reason=False)
deepeval_relevance = ContextualRelevancyMetric(model=JUDGE_MODEL, include_reason=False)

deepeval_scores = {}
for run in judged_runs:
    case = cases_by_id[run["case_id"]]
    test_case = LLMTestCase(input=case["input"], actual_output=run["answer"],
                            expected_output=case["reference_answer"], retrieval_context=run["contexts"])
    deepeval_scores[run["case_id"]] = {
        "precision": await deepeval_precision.a_measure(test_case, _show_indicator=False),
        "recall": await deepeval_recall.a_measure(test_case, _show_indicator=False),
        "relevance": await deepeval_relevance.a_measure(test_case, _show_indicator=False),
    }
print(f"✅ DeepEval judged {len(deepeval_scores)} cases")

✅ DeepEval judged 32 cases


In [16]:
# ============================================================================
# LABELS vs JUDGES, PER SLICE
# ============================================================================
def mean(values):
    return sum(values) / len(values)

print(f"{'slice':<10} {'n':>3} {'label recall@4':>15} | {'RAGAS P':>8} {'R':>5} {'Rel':>5} | "
      f"{'DeepEval P':>10} {'R':>5} {'Rel':>5}")
for slice_name in ["simple", "multi_hop"]:
    ids = [run["case_id"] for run in judged_runs if run["slice"] == slice_name]
    label_recall = []
    for case_id in ids:
        label_recall.append(recall_at_k(runs_by_id[case_id]["retrieved_chunk_ids"],
                                        cases_by_id[case_id]["relevant_chunk_ids"], 4))
    print(f"{slice_name:<10} {len(ids):>3} {mean(label_recall):>15.3f} | "
          f"{mean([ragas_scores[i]['precision'] for i in ids]):>8.3f} "
          f"{mean([ragas_scores[i]['recall'] for i in ids]):>5.3f} "
          f"{mean([ragas_scores[i]['relevance'] for i in ids]):>5.3f} | "
          f"{mean([deepeval_scores[i]['precision'] for i in ids]):>10.3f} "
          f"{mean([deepeval_scores[i]['recall'] for i in ids]):>5.3f} "
          f"{mean([deepeval_scores[i]['relevance'] for i in ids]):>5.3f}")
print("Reading: recall from labels and recall from judges answer the same question two ways, and agree.")
print("Precision is lower for multi_hop: its useful chunks are not always ranked first. DeepEval's")
print("relevance is low for simple questions because 3 of the 4 chunks are off-topic (see Part 5).")

slice        n  label recall@4 |  RAGAS P     R   Rel | DeepEval P     R   Rel
simple      19           1.000 |    0.930 0.947 1.000 |      1.000 1.000 0.221
multi_hop   13           1.000 |    0.838 0.974 1.000 |      0.908 0.910 0.490
Reading: recall from labels and recall from judges answer the same question two ways, and agree.
Precision is lower for multi_hop: its useful chunks are not always ranked first. DeepEval's
relevance is low for simple questions because 3 of the 4 chunks are off-topic (see Part 5).


---
## 🔎 Part 5: Where the labels and the judges disagree

An average hides the interesting cases. Two kinds of disagreement are worth reading one by one:

- **Labels say found, judge says missing**: every labelled chunk was retrieved, yet a judge
  thinks the context can't support the reference. Either the judge misread, or the reference
  answer says more than the labelled chunks do (a label problem).
- **Labels say missing, judge says found**: a labelled chunk was not retrieved, yet the context
  still supports the reference. Usually another chunk says the same thing: the label set was
  too narrow, not the retriever too weak.

In [17]:
# ============================================================================
# LIST EVERY DISAGREEMENT BETWEEN LABEL RECALL AND JUDGED RECALL
# ============================================================================
disagreements = []
for run in judged_runs:
    case_id = run["case_id"]
    label_recall = recall_at_k(run["retrieved_chunk_ids"], cases_by_id[case_id]["relevant_chunk_ids"], 4)
    judged_recall = min(ragas_scores[case_id]["recall"], deepeval_scores[case_id]["recall"])
    if label_recall == 1.0 and judged_recall < 0.5:
        disagreements.append((case_id, "labels: found, judge: missing", label_recall, judged_recall))
    if label_recall < 1.0 and judged_recall >= 0.9:
        disagreements.append((case_id, "labels: missing, judge: found", label_recall, judged_recall))

print(f"🔎 {len(disagreements)} disagreements out of {len(judged_runs)} judged cases")
for case_id, kind, label_recall, judged_recall in disagreements:
    case = cases_by_id[case_id]
    print(f"\n[{case_id}] {kind}  (label recall {label_recall:.2f}, lower judged recall {judged_recall:.2f})")
    print(f"   ❓ {case['input']}")
    print(f"   📝 reference: {case['reference_answer'][:200]}")
    print(f"   🏷️ labelled: {case['relevant_chunk_ids']}")
    print(f"   🔍 retrieved: {runs_by_id[case_id]['retrieved_chunk_ids']}")

🔎 1 disagreements out of 32 judged cases

[simple-h02] labels: found, judge: missing  (label recall 1.00, lower judged recall 0.00)
   ❓ Which countries do you ship to?
   📝 reference: Canada, the United Kingdom and the European Union, as well as domestically.
   🏷️ labelled: ['shipping-02']
   🔍 retrieved: ['shipping-02', 'shipping-01', 'orders-03', 'shipping-03']


When this notebook was last run, the one disagreement was `simple-h02`: its reference answer says
"as well as domestically", which no support article says. The judge was right and the label was
wrong: a reference must not claim more than the knowledge base, or every later correctness score
on that case punishes the agent for not inventing it. The fix belongs in the golden set
(`data/golden_set_handwritten.json`), followed by re-running notebook 03.

In [18]:
# ============================================================================
# DO THE TWO JUDGES AGREE WITH EACH OTHER?
# ============================================================================
for metric_name in ["precision", "recall", "relevance"]:
    gaps = []
    for case_id in ragas_scores:
        gaps.append(abs(ragas_scores[case_id][metric_name] - deepeval_scores[case_id][metric_name]))
    big_gaps = [gap for gap in gaps if gap >= 0.5]
    print(f"   {metric_name:<10} mean gap {mean(gaps):.3f}   cases with a gap ≥ 0.5: {len(big_gaps)} of {len(gaps)}")
print("Reading: precision and recall mostly agree. Relevance doesn't, because the two libraries define")
print("it differently: RAGAS asks whether the context as a whole is relevant, DeepEval counts the share")
print("of relevant statements. Same name, different metric: always report a judged score with its")
print("framework, and check judges against human labels (notebook 11) before trusting them.")

   precision  mean gap 0.086   cases with a gap ≥ 0.5: 2 of 32
   recall     mean gap 0.057   cases with a gap ≥ 0.5: 2 of 32
   relevance  mean gap 0.669   cases with a gap ≥ 0.5: 25 of 32
Reading: precision and recall mostly agree. Relevance doesn't, because the two libraries define
it differently: RAGAS asks whether the context as a whole is relevant, DeepEval counts the share
of relevant statements. Same name, different metric: always report a judged score with its
framework, and check judges against human labels (notebook 11) before trusting them.


---
## 📝 Summary

In this notebook, we learned:

### 1. The baseline run
- **One run, many notebooks**: `data/golden_runs.json` holds every golden case's route, steps,
  chunks, answer and end state; notebooks 04–09 score it

### 2. Label-based retrieval metrics
- **Five metrics, a few lines each**: hit rate, recall, precision, MRR, nDCG, per slice and per k
- **`ranx` agrees to the last decimal**, and `ranx.compare` adds a paired significance test:
  dense retrieval beats keyword overlap by more than chance

### 3. Judged retrieval metrics
- **No labels needed**: RAGAS and DeepEval judge precision, recall and relevance with a Qwen3 235B judge
- **Disagreements are the lesson**: they point at narrow labels, long references or a misreading judge

### Next Steps
- Notebook 05 judges the answers written from these chunks: faithfulness, relevancy and
  correctness, then splits each failure between the retriever and the generator